# Defi EquiAlgo - modele de depart (baseline)

**Hackathon Genie & Informatique 2026 - defi 24 h**

Ce carnet charge les donnees, entraine le modele actuellement en production chez
notre partenaire financier, mesure sa performance, et fait un premier audit d'equite.

Il ne corrige rien. C'est votre travail.

Executez-le en entier une fois (Cell > Run All), lisez la section 5, puis partez de la.

---


## 1. Chargement des donnees

Deux fichiers :

| Fichier | Lignes | Contient |
|---|---|---|
| `data/donnees_demandes.csv` | 10 000 | dossiers historiques **avec** `decision_octroi` |
| `data/candidats_evaluation.csv` | 4 000 | nouveaux dossiers, **sans** etiquette - c'est sur ceux-la que vous serez notes |


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    selection_rate,
    true_positive_rate,
)

demandes = pd.read_csv('data/donnees_demandes.csv')
candidats = pd.read_csv('data/candidats_evaluation.csv')

print(f'{len(demandes)} dossiers historiques, {len(candidats)} a evaluer')
demandes.head()


10000 dossiers historiques, 4000 a evaluer


,id_candidat,cote_r_equivalent,programme_etudes,region_administrative,code_postal_3,revenu_familial_estime,heures_travail_semaine,distance_domicile_campus_km,premiere_generation_universitaire,decision_octroi
0,C008033,25.96,Genie,Cote-Nord,G4R,17017,9,168.0,0,0
1,C004299,26.82,Arts et lettres,Cote-Nord,G5B,66820,22,99.3,0,0
2,C011788,27.37,Genie,Bas-Saint-Laurent,G5N,49223,19,364.1,0,0
3,C001144,21.23,Sciences,Gaspesie-Iles-de-la-Madeleine,G5C,90750,17,174.8,1,0
4,C003322,28.27,Sante,Bas-Saint-Laurent,G5N,36892,9,151.2,0,0


In [2]:
demandes.info()
print()
print(demandes.describe(include='all').T[['count', 'unique', 'top', 'mean']])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   id_candidat                        10000 non-null  object 
 1   cote_r_equivalent                  10000 non-null  float64
 2   programme_etudes                   10000 non-null  object 
 3   region_administrative              10000 non-null  object 
 4   code_postal_3                      10000 non-null  object 
 5   revenu_familial_estime             10000 non-null  int64  
 6   heures_travail_semaine             10000 non-null  int64  
 7   distance_domicile_campus_km        10000 non-null  float64
 8   premiere_generation_universitaire  10000 non-null  int64  
 9   decision_octroi                    10000 non-null  int64  
dtypes: float64(2), int64(4), object(4)
memory usage: 781.4+ KB

                                     count unique       top

## 2. Un premier regard sur les regions

Le comite a accorde une bourse a 40 % des candidats. Ce taux est-il le meme partout ?


In [3]:
# Les cinq regions administratives presentes dans le jeu de donnees.
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']

def groupe_region(df):
    """Regroupe les cinq regions en deux blocs : grands centres / regions eloignees."""
    return np.where(df['region_administrative'].isin(ELOIGNEES), 'Eloignee', 'Centre')

par_region = demandes.groupby('region_administrative').agg(
    candidats=('decision_octroi', 'size'),
    taux_octroi=('decision_octroi', 'mean'),
    cote_r_moyenne=('cote_r_equivalent', 'mean'),
).sort_values('taux_octroi')
print(par_region.round(3))

demandes['groupe'] = groupe_region(demandes)
print()
print(demandes.groupby('groupe')['decision_octroi'].mean().round(3))


                               candidats  taux_octroi  cote_r_moyenne
region_administrative                                                
Cote-Nord                           1178        0.263          27.325
Bas-Saint-Laurent                   1293        0.269          27.296
Gaspesie-Iles-de-la-Madeleine       1529        0.284          27.354
Montreal                            4001        0.483          27.988
Capitale-Nationale                  1999        0.484          27.984

groupe
Centre      0.484
Eloignee    0.273
Name: decision_octroi, dtype: float64


Notez l'ecart de cote R entre les regions, puis l'ecart de taux d'octroi.
Est-ce que le second s'explique entierement par le premier ? Gardez la question ouverte.


## 3. Le modele en production

Une foret aleatoire entrainee sur toutes les colonnes disponibles, y compris
`region_administrative`. C'est le modele que l'institution utilise aujourd'hui.


In [4]:
CATEGORIELLES = ['programme_etudes', 'region_administrative', 'code_postal_3']

def encoder(df_entrainement, df_cible):
    """Encode les colonnes categorielles et aligne les colonnes des deux tableaux.

    L'alignement (reindex) est important : si une categorie est absente d'un des
    deux jeux, get_dummies produit un nombre de colonnes different et le modele
    echoue - ou pire, apprend sur les mauvaises colonnes.
    """
    X = pd.get_dummies(
        df_entrainement.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    )
    Xc = pd.get_dummies(
        df_cible.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    ).reindex(columns=X.columns, fill_value=0)
    return X, Xc

# Decoupage interne pour mesurer honnetement la performance.
train, test = train_test_split(
    demandes, test_size=0.3, random_state=42, stratify=demandes['decision_octroi']
)
X_train, X_test = encoder(train, test)
y_train, y_test = train['decision_octroi'], test['decision_octroi']

modele = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)

print(f'Exactitude globale : {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, digits=3))


Exactitude globale : 88.13%

              precision    recall  f1-score   support

           0      0.890     0.916     0.903      1802
           1      0.867     0.830     0.848      1198

    accuracy                          0.881      3000
   macro avg      0.879     0.873     0.875      3000
weighted avg      0.881     0.881     0.881      3000



Environ 88 % d'exactitude. Du point de vue du tableau de bord trimestriel, le modele va tres bien.


## 4. Audit d'equite

Deux mesures, et elles ne disent pas la meme chose :

- **Parite demographique** - est-ce que les deux groupes recoivent des bourses au meme taux ?
- **Egalite des chances** - parmi les candidats qui meritent la bourse, est-ce que les deux
  groupes ont la meme chance de l'obtenir ?

La deuxieme demande de savoir qui merite la bourse. Regardez bien avec quoi on la calcule ici.


In [5]:
groupe_test = groupe_region(test)

cadre = MetricFrame(
    metrics={
        'exactitude': accuracy_score,
        'taux_selection': selection_rate,
        'taux_vrais_positifs': true_positive_rate,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=groupe_test,
)
print(cadre.by_group.round(4))

ecart_parite = demographic_parity_difference(y_test, y_pred, sensitive_features=groupe_test)
print(f'\nEcart de parite demographique : {ecart_parite:.4f}')


                     exactitude  taux_selection  taux_vrais_positifs
sensitive_feature_0                                                 
Centre                   0.8777          0.4586               0.8475
Eloignee                 0.8865          0.2710               0.7847

Ecart de parite demographique : 0.1876


Le `taux_vrais_positifs` ci-dessus est calcule par rapport a `decision_octroi`,
c'est-a-dire par rapport aux decisions passees du comite.

Autrement dit : on vient de verifier que le modele reproduit fidelement le comite.
Ce n'est pas la meme chose que verifier qu'il est juste.


## 5. Le piege

Le reflexe est de supprimer `region_administrative` et de declarer le probleme regle.
Mesurons.


In [6]:
essais = {'modele complet': []}

def entrainer_sans(colonnes_exclues, etiquette):
    X, Xt = encoder(train, test)
    garder = [c for c in X.columns if not c.startswith(tuple(colonnes_exclues))] if colonnes_exclues else list(X.columns)
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
    m.fit(X[garder], y_train)
    p = m.predict(Xt[garder])
    ecart = demographic_parity_difference(y_test, p, sensitive_features=groupe_test)
    print(f'{etiquette:<42} exactitude {accuracy_score(y_test, p):.4f}   ecart parite {ecart:.4f}')
    return ecart

entrainer_sans((), 'toutes les colonnes')
entrainer_sans(('region_administrative_',), 'sans region_administrative')
entrainer_sans(('region_administrative_', 'code_postal_3_'), 'sans region ni code postal')


toutes les colonnes                        exactitude 0.8813   ecart parite 0.1876
sans region_administrative                 exactitude 0.8837   ecart parite 0.1805
sans region ni code postal                 exactitude 0.8793   ecart parite 0.1731


0.17310721471687268

L'ecart bouge a peine.

D'autres colonnes portent l'information regionale : la distance domicile-campus, les
heures travaillees, le revenu familial, le code postal. Supprimer l'etiquette ne supprime
pas le signal. C'est l'**effet proxy**.

Mesurez-le vous-meme - une matrice de correlation, ou un modele qui essaie de predire la
region a partir des autres colonnes, vous dira lesquelles sont les plus bavardes.


## 6. Diagnostic regional

### 6.1. Disparites

Les taux historiques decrivent le comite; les TPR mesurent l'accord avec ses decisions. L'ecart de 0,270 contre la reference cachee n'est pas calculable ici. Chaque nouvelle cellule est autonome.

In [7]:
from IPython.display import display

d = pd.read_csv('data/donnees_demandes.csv')
d['groupe'] = np.where(d.region_administrative.isin(['Montreal', 'Capitale-Nationale']), 'Centre', 'Eloignee')
for group in ['region_administrative', 'groupe']:
    display(d.groupby(group).agg(n=('decision_octroi', 'size'), taux_octroi=('decision_octroi', 'mean')))

tr, te = train_test_split(d, test_size=0.3, random_state=42, stratify=d.decision_octroi)
drop = ['id_candidat', 'decision_octroi', 'groupe']
cats = ['programme_etudes', 'region_administrative', 'code_postal_3']
xtr = pd.get_dummies(tr.drop(columns=drop), columns=cats)
xte = pd.get_dummies(te.drop(columns=drop), columns=cats).reindex(columns=xtr.columns, fill_value=0)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
pred = rf.fit(xtr, tr.decision_octroi).predict(xte)
metrics = MetricFrame(
    metrics={'accord_historique': accuracy_score, 'selection': selection_rate, 'TPR_historique': true_positive_rate},
    y_true=te.decision_octroi, y_pred=pred, sensitive_features=te.groupe,
).by_group
display(metrics.round(4))
rates = d.groupby('groupe').decision_octroi.mean()
display(pd.Series({
    'Octroi historique': rates['Centre'] - rates['Eloignee'],
    'Selection du modele': metrics.loc['Centre', 'selection'] - metrics.loc['Eloignee', 'selection'],
    'TPR historique': metrics.loc['Centre', 'TPR_historique'] - metrics.loc['Eloignee', 'TPR_historique'],
}, name='Ecart Centre - Eloignee').round(4))

,n,taux_octroi
region_administrative,,
Bas-Saint-Laurent,1293,0.269142
Capitale-Nationale,1999,0.484242
Cote-Nord,1178,0.263158
Gaspesie-Iles-de-la-Madeleine,1529,0.283846
Montreal,4001,0.483379


,n,taux_octroi
groupe,,
Centre,6000,0.483667
Eloignee,4000,0.273000


,accord_historique,selection,TPR_historique
groupe,,,
Centre,0.8777,0.4586,0.8475
Eloignee,0.8865,0.2710,0.7847


Octroi historique      0.2107
Selection du modele    0.1876
TPR historique         0.0628
Name: Ecart Centre - Eloignee, dtype: float64

### 6.2. Associations avec la region

Exploration sur les 70 % d'entrainement uniquement. Eloignee = 1, Centre = 0. Correlation point-biseriale pour les mesures numeriques; V de Cramer pour les categories. Une faible correlation n'exclut pas un signal non lineaire.

In [8]:
from scipy.stats import pointbiserialr
from scipy.stats.contingency import association

d = pd.read_csv('data/donnees_demandes.csv')
y = (~d.region_administrative.isin(['Montreal', 'Capitale-Nationale'])).astype(int)
tr, _, g, _ = train_test_split(d, y, test_size=0.3, random_state=42, stratify=y)
numeric = ['cote_r_equivalent', 'revenu_familial_estime', 'distance_domicile_campus_km', 'heures_travail_semaine']
display(pd.Series({c: pointbiserialr(g, tr[c]).statistic for c in numeric}, name='r (Eloignee = 1)').sort_values(key=abs, ascending=False).round(3))
display(tr.assign(groupe=g.map({0: 'Centre', 1: 'Eloignee'})).groupby('groupe')[numeric].agg(['mean', 'median', 'std']).round(2).T)

categorical = ['programme_etudes', 'code_postal_3', 'premiere_generation_universitaire']
tables = {c: pd.crosstab(tr[c], g.rename('Eloignee')) for c in categorical}
display(pd.Series({c: association(t, method='cramer') for c, t in tables.items()}, name='V de Cramer').sort_values(ascending=False).round(3))
display(pd.concat({c: t.rename(columns={0: 'Centre', 1: 'Eloignee'}) for c, t in tables.items()}, names=['variable', 'valeur']))

distance_domicile_campus_km    0.815
heures_travail_semaine         0.524
revenu_familial_estime        -0.296
cote_r_equivalent             -0.113
Name: r (Eloignee = 1), dtype: float64

groupe                                Centre  Eloignee
cote_r_equivalent           mean       28.03     27.34
                            median     28.01     27.34
                            std         2.99      2.98
revenu_familial_estime      mean    76028.58  56529.90
                            median  68783.00  51405.50
                            std     33779.08  25783.59
distance_domicile_campus_km mean       19.86    221.23
                            median     17.10    204.35
                            std        13.27    109.75
heures_travail_semaine      mean        8.97     13.05
                            median      9.00     13.00
                            std         2.95      3.65

code_postal_3                        1.000
premiere_generation_universitaire    0.192
programme_etudes                     0.094
Name: V de Cramer, dtype: float64

Eloignee                                             Centre  Eloignee
variable                          valeur                             
programme_etudes                  Arts et lettres       649       565
                                  Genie                1056       536
                                  Sante                 723       531
                                  Sciences              948       551
                                  Sciences sociales     824       617
code_postal_3                     G0C                     0       401
                                  G0G                     0       277
                                  G0L                     0       286
                                  G1R                   347         0
                                  G1V                   382         0
                                  G2E                   346         0
                                  G3A                   337         0
                                  G4R                     0       279
                                  G4X                     0       341
                                  G5B                     0       264
                                  G5C                     0       343
                                  G5L                     0       313
                                  G5N                     0       296
                                  H1A                   571         0
                                  H2X                   566         0
                                  H3T                   560         0
                                  H4L                   563         0
                                  H8N                   528         0
premiere_generation_universitaire 0                    3110      1556
                                  1                    1090      1244

### 6.3. Prediction de la region

Cible : Eloignee = 1. Exclusion de la region, des identifiants et de la decision historique. Memes partitions pour tous les essais fixes : validation croisee stratifiee a 5 plis sur 70 %, puis test reserve de 30 %. Encodage appris dans chaque pli.

Les variables seules, combinaisons et retraits revelent les signaux redondants. Reference : accuracy majoritaire = 60 %, balanced accuracy et AUC = 0,5. IC a 95 % par 500 bootstraps stratifies du test, conditionnels aux modeles ajustes.

In [9]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

d = pd.read_csv('data/donnees_demandes.csv')
y = (~d.region_administrative.isin(['Montreal', 'Capitale-Nationale'])).astype(int)
x = d.drop(columns=['id_candidat', 'region_administrative', 'decision_octroi'])
xtr, xte, ytr, yte = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)
subsets = {'Reference': list(x), **{c: [c] for c in x}, 'Toutes': list(x)}
subsets['Sans postal'] = [c for c in x if c != 'code_postal_3']
subsets['Sans postal ni distance'] = [c for c in x if c not in ['code_postal_3', 'distance_domicile_campus_km']]
subsets['Distance + revenu + heures'] = ['distance_domicile_campus_km', 'revenu_familial_estime', 'heures_travail_semaine']
subsets['Cote R + programme + premiere generation'] = ['cote_r_equivalent', 'programme_etudes', 'premiere_generation_universitaire']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores, fitted = [], {}
for name, cols in subsets.items():
    encoder_proxy = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore'), make_column_selector(dtype_include=object)),
    ], remainder='passthrough')
    estimator = DummyClassifier(strategy='prior') if name == 'Reference' else RandomForestClassifier(
        n_estimators=100, min_samples_leaf=10, random_state=42,
    )
    pipe = make_pipeline(encoder_proxy, estimator)
    result = cross_validate(pipe, xtr[cols], ytr, cv=cv, scoring={'BA': 'balanced_accuracy', 'AUC': 'roc_auc'})
    scores.append({'variables': name, **{
        f'{metric} CV': f'{result[f"test_{metric}"].mean():.3f} +/- {result[f"test_{metric}"].std(ddof=1):.3f}'
        for metric in ['BA', 'AUC']
    }})
    fitted[name] = pipe.fit(xtr[cols], ytr)
display(pd.DataFrame(scores).set_index('variables'))

rng = np.random.default_rng(42)
yt = yte.to_numpy()
groups = [np.flatnonzero(yt == g) for g in [0, 1]]
resamples = [np.concatenate([rng.choice(g, len(g), replace=True) for g in groups]) for _ in range(500)]
results = []
for name, pipe in fitted.items():
    probability = pipe.predict_proba(xte[subsets[name]])[:, 1]
    prediction = (probability > 0.5).astype(int)
    row = {'variables': name}
    for label, metric, values in [('BA', balanced_accuracy_score, prediction), ('AUC', roc_auc_score, probability)]:
        low, high = np.quantile([metric(yt[i], values[i]) for i in resamples], [0.025, 0.975])
        row[label] = metric(yt, values)
        row[f'{label} IC95'] = f'[{low:.3f}, {high:.3f}]'
    results.append(row)
display(pd.DataFrame(results).set_index('variables').round(3))

,BA CV,AUC CV
variables,,
Reference,0.500 +/- 0.000,0.500 +/- 0.000
cote_r_equivalent,0.517 +/- 0.015,0.529 +/- 0.020
programme_etudes,0.500 +/- 0.000,0.553 +/- 0.011
code_postal_3,1.000 +/- 0.000,1.000 +/- 0.000
revenu_familial_estime,0.592 +/- 0.021,0.649 +/- 0.014
heures_travail_semaine,0.715 +/- 0.011,0.806 +/- 0.008
distance_domicile_campus_km,0.982 +/- 0.003,0.995 +/- 0.002
premiere_generation_universitaire,0.592 +/- 0.011,0.592 +/- 0.011
Toutes,0.989 +/- 0.003,1.000 +/- 0.000


,BA,BA IC95,AUC,AUC IC95
variables,,,,
Reference,0.500,"[0.500, 0.500]",0.500,"[0.500, 0.500]"
cote_r_equivalent,0.505,"[0.490, 0.520]",0.514,"[0.493, 0.535]"
programme_etudes,0.500,"[0.500, 0.500]",0.557,"[0.535, 0.579]"
code_postal_3,1.000,"[1.000, 1.000]",1.000,"[1.000, 1.000]"
revenu_familial_estime,0.602,"[0.583, 0.619]",0.653,"[0.632, 0.673]"
heures_travail_semaine,0.702,"[0.687, 0.719]",0.802,"[0.786, 0.817]"
distance_domicile_campus_km,0.982,"[0.977, 0.987]",0.997,"[0.995, 0.999]"
premiere_generation_universitaire,0.583,"[0.566, 0.600]",0.583,"[0.566, 0.600]"
Toutes,0.990,"[0.986, 0.993]",1.000,"[1.000, 1.000]"


Sur le test, le postal seul atteint une balanced accuracy de 100 %, la distance seule 98,2 %, les variables sans postal 98,6 % et sans postal ni distance 75,8 %. La cote R seule atteint 50,5 % : son signal predictif est faible ici.

La predictibilite mesure l'information regionale disponible, pas son utilisation par le comite ni un effet causal sur l'octroi. Le role causal de la cote R et l'equite contre la reference cachee restent inconnus. Le test mesure la generalisation a des candidats de la meme distribution, pas a de nouvelles zones postales.

---

### Pistes pour la suite

- Quelles colonnes predisent le mieux la region ? Ce sont vos proxys.
- Entre parite demographique et egalite des chances, laquelle defendriez-vous devant un
  comite d'ethique, et pourquoi ? Les deux ne peuvent pas etre satisfaites en meme temps
  si les deux groupes n'ont pas exactement le meme profil.
- `fairlearn.postprocessing.ThresholdOptimizer` et `fairlearn.reductions.ExponentiatedGradient`
  sont les deux outils les plus rapides a mettre en place. Le premier ajuste les seuils apres
  coup, le second reentraine sous contrainte.
- Faites varier la contrainte d'equite et tracez le nuage (equite, exactitude) : c'est votre
  front de Pareto, et le jury va vous demander de le defendre.
- Si `decision_octroi` est une etiquette biaisee, est-ce que le meilleur modele est celui qui
  la predit le mieux ?


## 7. Votre mission

Produire, pour les 4 000 candidats de `data/candidats_evaluation.csv`, une decision
d'octroi plus juste que celle du modele actuel.

Trois contraintes :

1. **Budget fixe.** L'institution finance un nombre fixe de bourses. Votre taux d'octroi
   global doit rester entre **36 % et 44 %**. Une soumission hors de cette plage recoit 0
   sur la section technique. Accorder la bourse a tout le monde n'est pas une solution.
2. **Vous n'etes pas notes sur `decision_octroi`.** Le jury compare vos decisions a un
   etalon de reference construit independamment du comite historique, que vous n'avez pas.
   Reproduire parfaitement `decision_octroi` reproduit parfaitement le biais.
3. **Format.** Deux colonnes, 4 000 lignes, valeurs 0 ou 1.

La cellule suivante ecrit une soumission valide a partir du modele biaise. Elle sert de
gabarit. Remplacez `predictions` par les votres.


In [10]:
historique = pd.read_csv('data/donnees_demandes.csv')
candidats_finaux = pd.read_csv('data/candidats_evaluation.csv')
variables = ['cote_r_equivalent', 'programme_etudes', 'revenu_familial_estime',
             'heures_travail_semaine', 'distance_domicile_campus_km',
             'premiere_generation_universitaire']
preparation = ColumnTransformer([
    ('programme', OneHotEncoder(handle_unknown='ignore'), ['programme_etudes']),
], remainder='passthrough')
modele_octroi = make_pipeline(preparation, RandomForestClassifier(
    n_estimators=300, min_samples_leaf=20, random_state=42,
))
modele_octroi.fit(historique[variables], historique.decision_octroi)
scores_octroi = modele_octroi.predict_proba(candidats_finaux[variables])[:, 1]

regions_valides = ['Montreal', 'Capitale-Nationale', 'Bas-Saint-Laurent',
                   'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']
assert candidats_finaux.region_administrative.isin(regions_valides).all()
groupes_finaux = (~candidats_finaux.region_administrative.isin(regions_valides[:2])).astype(int)
effectifs = groupes_finaux.value_counts().sort_index()
quotas_exacts = 0.40 * effectifs
quotas = np.floor(quotas_exacts).astype(int)
reste = int(round(0.40 * len(candidats_finaux))) - int(quotas.sum())
quotas.loc[(quotas_exacts - quotas).sort_values(ascending=False).index[:reste]] += 1

predictions_finales = np.zeros(len(candidats_finaux), dtype=int)
departage = pd.util.hash_pandas_object(candidats_finaux.id_candidat, index=False).to_numpy()
for groupe, quota in quotas.items():
    indices = np.flatnonzero(groupes_finaux.to_numpy() == groupe)
    ordre = np.lexsort((departage[indices], -scores_octroi[indices]))
    predictions_finales[indices[ordre[:quota]]] = 1

soumission = pd.DataFrame({'id_candidat': candidats_finaux.id_candidat,
                          'decision_octroi': predictions_finales})
assert len(soumission) == 4000 and soumission.id_candidat.is_unique
assert soumission.id_candidat.notna().all() and soumission.decision_octroi.isin([0, 1]).all()
assert soumission.id_candidat.equals(candidats_finaux.id_candidat)
assert soumission.decision_octroi.sum() == 1600
assert all(abs(predictions_finales[groupes_finaux == g].mean() - 0.40) <= 1 / n
           for g, n in effectifs.items())
soumission.to_csv('predictions.csv', index=False)
print(f'{len(soumission)} predictions, taux octroi : {soumission.decision_octroi.mean():.1%}')


4000 predictions, taux octroi : 40.0%
